# TAREA 5.5: Sistema experto basado en probabilidad para diagnóstico industrial
**Módulo:** Modelos de Inteligencia Artificial  
**Curso:** Especialización en IA y Big Data  


---

## 1. Introducción
En esta actividad implementarás un **Sistema experto basado en el conocimiento (SBC)** con un enfoque probabilístico. El sistema debe ayudar a detectar averías en maquinaria industrial analizando:
1. **Síntomas** detectados por el operario.
2. **Probabilidades históricas** (Base de Conocimiento).
3. **Estado de desgaste** de las piezas (Horas de uso).

El sistema debe ser capaz de **aprender** de cada diagnóstico confirmado por un técnico humano, mejorando sus predicciones futuras.

In [3]:
!pip install pandas -q

In [1]:
import json
import os
import pandas as pd

class SistemaExpertoAverias:
    FACTOR_DESGASTE = 1.5
    INCREMENTO_APRENDIZAJE = 0.05
    PROB_MAX = 0.95
    PROB_MIN = 0.01

    def __init__(self, archivo_db='base_conocimiento.json'):
        self.archivo_db = archivo_db
        self.datos = self.cargar_datos()

    def cargar_datos(self):
        """Carga la base de conocimiento desde JSON o crea una por defecto."""
        if os.path.exists(self.archivo_db):
            with open(self.archivo_db, 'r', encoding='utf-8') as f:
                return json.load(f)

        # Datos iniciales (Matriz de probabilidad y estado de piezas)
        return {
            "piezas": {
                "Motor":  {"horas_limite": 5000, "horas_actuales": 4800, "fallos_acumulados": 0},
                "Correa": {"horas_limite": 1000, "horas_actuales": 950,  "fallos_acumulados": 0},
                "Filtro": {"horas_limite": 2000, "horas_actuales": 100,  "fallos_acumulados": 0}
            },
            "probabilidades": {
                "Vibracion":          {"Motor": 0.6, "Correa": 0.3, "Filtro": 0.1},
                "Ruido_Agudo":        {"Motor": 0.2, "Correa": 0.7, "Filtro": 0.1},
                "Sobrecalentamiento": {"Motor": 0.7, "Correa": 0.1, "Filtro": 0.2}
            }
        }

    def guardar_datos(self):
        """Persistencia en disco."""
        with open(self.archivo_db, 'w', encoding='utf-8') as f:
            json.dump(self.datos, f, indent=4, ensure_ascii=False)

    def diagnosticar(self, sintoma):
        """Motor de Inferencia Probabilístico con factor de desgaste y normalización."""
        if sintoma not in self.datos["probabilidades"]:
            return None

        prob_sintoma = self.datos["probabilidades"][sintoma]
        resultados = []

        for pieza, prob_base in prob_sintoma.items():
            info = self.datos["piezas"][pieza]
            # Factor x1.5 si la pieza ha SUPERADO su límite de horas
            supera_limite = info["horas_actuales"] > info["horas_limite"]
            factor = self.FACTOR_DESGASTE if supera_limite else 1.0
            prob_ajustada = prob_base * factor
            resultados.append({
                "Pieza": pieza,
                "Prob_Base": round(prob_base, 3),
                "Desgaste": f"{info['horas_actuales']}/{info['horas_limite']}",
                "Factor": factor,
                "Prob_Calculada": round(prob_ajustada, 3)
            })

        # Normalización: la salida es una distribución válida (suma = 1)
        total = sum(r["Prob_Calculada"] for r in resultados)
        if total > 0:
            for r in resultados:
                r["Prob_Normalizada"] = round(r["Prob_Calculada"] / total, 3)

        return sorted(resultados, key=lambda x: x["Prob_Normalizada"], reverse=True)

    def retroalimentacion(self, sintoma, pieza_real):
        """Aprendizaje: +5% a la pieza confirmada, redistribuye el resto, resetea horas."""
        if sintoma not in self.datos["probabilidades"]:
            raise ValueError(f"Síntoma '{sintoma}' no registrado.")
        if pieza_real not in self.datos["piezas"]:
            raise ValueError(f"Pieza '{pieza_real}' no registrada.")

        probs = self.datos["probabilidades"][sintoma]

        # Incremento acotado
        nuevo_valor = min(self.PROB_MAX, probs[pieza_real] + self.INCREMENTO_APRENDIZAJE)
        delta = nuevo_valor - probs[pieza_real]
        probs[pieza_real] = nuevo_valor

        # Redistribución proporcional para mantener suma = 1
        otras = {p: v for p, v in probs.items() if p != pieza_real}
        suma_otras = sum(otras.values())
        if suma_otras > 0 and delta > 0:
            for p in otras:
                reduccion = delta * (otras[p] / suma_otras)
                probs[p] = max(self.PROB_MIN, probs[p] - reduccion)

        # Reset de la pieza reparada
        self.datos["piezas"][pieza_real]["horas_actuales"] = 0
        self.datos["piezas"][pieza_real]["fallos_acumulados"] += 1

        self.guardar_datos()
        print(f"\n[SISTEMA] Aprendizaje completado. P({pieza_real}|{sintoma}) = {nuevo_valor:.3f}")

In [2]:
# Inicializar el sistema
sistema = SistemaExpertoAverias()

print("SISTEMA EXPERTO DE DIAGNÓSTICO INDUSTRIAL")
print("------------------------------------------")
print(f"Síntomas disponibles: {', '.join(sistema.datos['probabilidades'].keys())}")

sintoma_input = input("Introduzca el síntoma observado: ").strip()
diagnostico = sistema.diagnosticar(sintoma_input)

if diagnostico:
    print("\nResultados del Motor de Inferencia:")
    df_res = pd.DataFrame(diagnostico)
    print(df_res.to_string(index=False))

    print("\n--- VALIDACIÓN DEL TÉCNICO ---")
    confirmacion = input(f"¿Qué pieza causó realmente la avería? ({', '.join(sistema.datos['piezas'].keys())}): ").strip()
    try:
        sistema.retroalimentacion(sintoma_input, confirmacion)
    except ValueError as e:
        print(f"[ERROR] {e}")
else:
    print("Síntoma no registrado en la base de conocimiento.")

SISTEMA EXPERTO DE DIAGNÓSTICO INDUSTRIAL
------------------------------------------
Síntomas disponibles: Vibracion, Ruido_Agudo, Sobrecalentamiento

Resultados del Motor de Inferencia:
 Pieza  Prob_Base  Desgaste  Factor  Prob_Calculada  Prob_Normalizada
Correa        0.7  950/1000     1.0             0.7               0.7
 Motor        0.2 4800/5000     1.0             0.2               0.2
Filtro        0.1  100/2000     1.0             0.1               0.1

--- VALIDACIÓN DEL TÉCNICO ---

[SISTEMA] Aprendizaje completado. P(Correa|Ruido_Agudo) = 0.750


In [3]:
# Celda para que el alumno vea cómo cambia el JSON visualmente
print("ESTADO ACTUAL DE LA BASE DE CONOCIMIENTO (JSON)")
df_piezas = pd.DataFrame(sistema.datos["piezas"]).T
df_piezas["% desgaste"] = (df_piezas["horas_actuales"] / df_piezas["horas_limite"] * 100).round(1)
display(df_piezas)

print("\nMATRIZ DE PROBABILIDADES ACTUALIZADA")
df_probs = pd.DataFrame(sistema.datos["probabilidades"])
df_probs.loc["TOTAL"] = df_probs.sum()
display(df_probs.round(3))

ESTADO ACTUAL DE LA BASE DE CONOCIMIENTO (JSON)


,horas_limite,horas_actuales,fallos_acumulados,% desgaste
Motor,5000,4800,0,96.0
Correa,1000,0,1,0.0
Filtro,2000,100,0,5.0



MATRIZ DE PROBABILIDADES ACTUALIZADA


,Vibracion,Ruido_Agudo,Sobrecalentamiento
Motor,0.6,0.167,0.7
Correa,0.3,0.750,0.1
Filtro,0.1,0.083,0.2
TOTAL,1.0,1.000,1.0


In [4]:
# Prueba explícita del factor de desgaste: forzamos al Motor a superar su límite
sistema.datos["piezas"]["Motor"]["horas_actuales"] = 5200
sistema.guardar_datos()

print("Diagnóstico con Motor en sobreuso (5200/5000) ante 'Sobrecalentamiento':")
resultado = sistema.diagnosticar("Sobrecalentamiento")
print(pd.DataFrame(resultado).to_string(index=False))

Diagnóstico con Motor en sobreuso (5200/5000) ante 'Sobrecalentamiento':
 Pieza  Prob_Base  Desgaste  Factor  Prob_Calculada  Prob_Normalizada
 Motor        0.7 5200/5000     1.5            1.05             0.778
Filtro        0.2  100/2000     1.0            0.20             0.148
Correa        0.1    0/1000     1.0            0.10             0.074


## Recursos y Enlaces de Interés
* **Conceptos:** [Sistemas Expertos Probabilísticos y Redes Bayesianas](https://iturbide.org/sistemas-expertos-probabilisticos/)
* **Teoría:** [Teorema de Bayes explicado de forma sencilla](https://recursos.citic.es/ia/bayes)
* **Python:** [Documentación oficial de la librería JSON](https://docs.python.org/3/library/json.html)
* **Herramienta sugerida:** [Netica (para modelado visual de probabilidades)](https://www.norsys.com/netica.html)